# Face Detection Model Training

# Step 1: Dependencies installation

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.3 MB/s eta 0:00:00


# Step 2: Google Drive mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Step 3: Dataset Preparation

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/COS40007_Project/face_detection/det_batch_0.zip"
extract_path = "/content/drive/MyDrive/COS40007_Project/face_detection/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)

Extracted to: /content/drive/MyDrive/COS40007_Project/face_detection/dataset


# Step 4: Dataset splitting

In [ ]:
import os
import random
import shutil
from pathlib import Path

random.seed(42)

img_dir = Path("/content/drive/MyDrive/COS40007_Project/face_detection/dataset/det_batch_0/images")
label_dir = Path("/content/drive/MyDrive/COS40007_Project/face_detection/dataset/det_batch_0/labels")

output_dir = Path("/content/drive/MyDrive/COS40007_Project/face_detection/dataset_train")

train_img = output_dir/"images/train"
val_img   = output_dir/"images/val"
test_img  = output_dir/"images/test"

train_lbl = output_dir/"labels/train"
val_lbl   = output_dir/"labels/val"
test_lbl  = output_dir/"labels/test"

for p in [train_img, val_img, test_img, train_lbl, val_lbl, test_lbl]:
    p.mkdir(parents=True, exist_ok=True)

images = list(img_dir.glob("*.*"))
random.shuffle(images)

n = len(images)
train_split = int(0.8 * n)
val_split = int(0.9 * n)

train_files = images[:train_split]
val_files   = images[train_split:val_split]
test_files  = images[val_split:]

def copy_files(files, img_out, lbl_out):
    for img_path in files:
        lbl_path = label_dir / (img_path.stem + ".txt")

        shutil.copy(img_path, img_out / img_path.name)

        if lbl_path.exists():
            shutil.copy(lbl_path, lbl_out / lbl_path.name)

copy_files(train_files, train_img, train_lbl)
copy_files(val_files, val_img, val_lbl)
copy_files(test_files, test_img, test_lbl)

print("Split done!")
print(len(train_files), len(val_files), len(test_files))

Split done!
239 30 30


# Step 5: Create training YAML script

In [ ]:
yaml_content = """
path: /content/drive/MyDrive/COS40007_Project/face_detection/dataset_train

train: images/train
val: images/val
test: images/test

nc: 1
names: ["face"]
"""

with open("/content/drive/MyDrive/COS40007_Project/face_detection/data.yaml", "w") as f:
    f.write(yaml_content)

print("data.yaml created")

data.yaml created


# Step 6: Model Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/drive/MyDrive/COS40007_Project/face_detection/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,

    # ---- LIGHT AUGMENTATION ----
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    degrees=5,
    translate=0.05,
    scale=0.3,
    shear=0.0,

    mosaic=0.5,
    mixup=0.0,

    fliplr=0.5,
    flipud=0.0,

    # ---- SAVE TO GOOGLE DRIVE ----
    project="/content/drive/MyDrive/COS40007_Project/face_detection/runs",
    name="yolov8_face",
    exist_ok=True
)



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/COS40007_Project/face_detection/data.yaml, degrees=5, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_

# Step 7: Model Evaluation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

run_dir = Path("/content/drive/MyDrive/COS40007_Project/face_detection/runs/yolov8_face")

csv_path = run_dir / "results.csv"

df = pd.read_csv(csv_path)

# YOLOv8 columns:
# train/box_loss, val/box_loss, metrics/mAP50-95(B), etc.

plt.figure()
plt.plot(df["epoch"], df["train/box_loss"], label="train box loss")
plt.plot(df["epoch"], df["val/box_loss"], label="val box loss")
plt.legend()
plt.title("Box Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

plt.figure()
plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@50")
plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP@50-95")
plt.legend()
plt.title("Validation mAP")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.show()

In [ ]:
model = YOLO("runs/detect/train/weights/best.pt")
model.predict("/content/test.jpg", save=True)